# Vision
### Idea
<!-- What would you like to try out? -->
* We would like to create a song recommender using the Spotify API based on a user's song history. We would like to create a new metric for determing which songs to suggest. 

### Vision
<!-- What would be an ideal, big-picture conclusion?  -->
* An idea conclusion would be that our application produces a song that makes sense/the user likes 80% of the time. If we have time, we would like to make a simple web app to host our app. 

### Feasible Set
<!-- What would be a reasonable final milestone for cs181y? -->
* A reasonable milestone would be to have the basic functionality of the app working, where a user gets song suggestions. 

# Starting Points
### Walkthroughs/Guides
<!-- Are you basing your project on one or more on-line guides?  Include links and, briefly, what those walkthroughs achieve, and then… -->
* We are uses this tutorial as a starting point for our project: [Spotify Recommendation System with Machine Learning](https://thecleverprogrammer.com/2021/03/03/spotify-recommendation-system-with-machine-learning/)
    * This tutorial uses a k-means clustering algorithm to create categories for songs.
    * This tutorial finds the distance between each song it wants to recommend and all the other songs in the dataset and gives the top N songs as recommendations.
* [Inside Spotify’s Recommender System: A Complete Guide to Spotify Recommendation Algorithms](https://www.music-tomorrow.com/blog/how-spotify-recommendation-system-works-a-complete-guide-2022) explains the different features that Spotify uses.

### Beyond Walkthroughs
<!-- how are you imagining personalizing that starting point?! -->
* The tutorial we found uses datasets that the author downloaded. We hope to modify our app to retrieve live data from the API. 

### Libraries
<!-- Have you been able to install/access libraries you need or you want to explore? -->
* We will be using many of the libraries we have used before for homework:
    * numpy
    * pandas
    * seaborn
    * matplotlib
    * math
* tqdm - a progress bar generator

### Your Access 
<!-- Software-wise, what have you tried? What works? What does not? -->
* We have previously used the Spotify API for homework. We were able to retrieve lots of different information about artists, songs, and the user's Spotify history/profile. 

### Datasets
<!-- Data-wise, what data do you plan to work with?  Have you been able to read and show simple/naive access to the data? -->
* We will not be using a dataset. Instead, we will be pulling user info and songs/artists info from the Spotify API.

# Progress
### Your Software
<!-- What software have you gotten to work so far?  -->
* We were able to follow the tutorial and replicate the results using the dataset they provided.

### Capabilities
<!-- What capabilities does it have so far? Give an overview of how it is doing (whatever that might be!) -->
* The recommender system is able to take in a song and find the top n songs that are closest to the inputted song and return them.

### Progress Overall
<!-- How smoothly has your initial explorations progressed? -->
* The progress has been smooth so far, because we have been replicating ideas that we or someone else have gotten to work before.

### Challenges
<!-- What resources have proven more difficult than you expected? Are there any that have proven less difficult? -->
* So far, we have not had any challenges yet.

### Missing Capabilities
<!-- 
What have you worked on that is not-yet-functional or not-yet-complete or not-yet-started? 
Are the next steps straightforward or is there a risk that something will require a change in direction -- or more exploration. 
(Note: it's no problem if that's the case -- with software it's always worth asking whether 
    (a) adding more "force" (hours/concentration/etc) or
    (b) adding more "perspective" would be a better next step! It's often (b)! -->
* We have not yet integrated the API code with the recommender system.

# Presentation
[Presentation](https://docs.google.com/presentation/d/1MT9_OQ1_gd9h1YJQV-jE24P-hraiUTrFQXJhWeaxKVg/edit)
<!-- Also, you should create a first-draft of a short (5-6 minute) Google-slides presentation, which should include
[vision]    slide(s) on the project's goal and vision  (~1 minute)
[starting points]    slide(s) on the project's resources  (~½ to 1 minute)
[progress]    slide(s) on the project's progress with "demos"  (~1-3 minutes)
[what's next]    slide(s) on what's still to go…  (~½ to 1 minute)
include a link to your presentation in your submission!   (be sure it's accessible.)

That "missing capabilities" section leads naturally to "What's next?" -->



# What's Next

### Looking Ahead
<!-- Has the week's experience changed your project plan? If so, how? -->
* We do not forsee any plan changes yet.

### Next Steps
<!-- What is your break down / plan for the week to come? -->
* We will continue to add the API capabilities to the system.

### Overall Goals
<!-- How have you adapted your overall project goals from this week's experiences? -->
* We defined how we would want the final results to look like.

### Reflection
<!-- If you could re-run this week in your project, are there things you'd do differently? What are your thoughts on the process thus far? -->

<!-- And, it's open-ended from there…  More is welcome, not expected or needed :-)

In essence, the goal here is to (a) make sure you have continued to explore your project space -- and made progress, even if it's progress of a different kind than you were expecting -- and (b) that you are not only exploring  successfully started using it -- and have a concrete plan for what to pursue next.

Remember to submit some of your source-code progress so far (if it's not already there in the notebook, in whatever your format is…), as well… -->
* We think that the work we have done so far was good progress, so we don't plan on doing anything differently.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
sns.set()

In [2]:
# Helper functions
import spotipy
# from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.oauth2 import SpotifyOAuth
import time 

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id="76f8001ab55843ac979a4728d3ed78c8",
                                                client_secret="8d93518878eb4da09dff1123fb4f73ac",
                                                redirect_uri="http://localhost:8888/callback",
                                                scope="user-top-read"))



In [3]:
def find_top_artists():
    '''
        Output: finds user's top artist for three time periods (short term, medium term, long term)
    '''
    artists = {}

    for sp_range in ['short_term', 'medium_term', 'long_term']:
        results = sp.current_user_top_artists(time_range=sp_range, limit=1)
        if results:
            artists[sp_range] = results['items'][0]['name']
        time.sleep(0.1)
    return artists


def get_artist_uri(name):
    '''
        Input: artist name
        Output: artist uri
    '''
    results = sp.search(q='artist:' + name, type='artist')
    items = results['artists']['items']
    if len(items) > 0 and items[0]["uri"]:
        return items[0]["uri"]
    else:
        return None


def find_artist_top_song(artist_uri):
    '''
        Input: artist uri
        Output: the artist's most popular song
    '''
    results = sp.artist_top_tracks(artist_uri, country="US")
    
    if results and results["tracks"] and results["tracks"][0] and results["tracks"][0]["name"] and results["tracks"][0]["id"]:
        return (results["tracks"][0]["name"], results["tracks"][0]["id"])

def find_song_danceability(song_id):
    '''
        Intput: a song id
        Output: the danceability score of the song
    '''
    results = sp.audio_features(song_id)
    if results and results[0]["danceability"]:
        return results[0]["danceability"]

In [4]:
# Retrieve data
data = pd.read_csv("spotify.csv")
data.head()

,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo,valence,year
0,0.991000,['Mamie Smith'],0.598,168333,0.224,0,0cS0A1fUEUd1EW3FcF8AEI,0.000522,5,0.3790,-12.628,0,Keep A Song In Your Soul,12,1920,0.0936,149.976,0.6340,1920
1,0.643000,"[""Screamin' Jay Hawkins""]",0.852,150200,0.517,0,0hbkKFIJm7Z05H8Zl9w30f,0.026400,5,0.0809,-7.261,0,I Put A Spell On You,7,1920-01-05,0.0534,86.889,0.9500,1920
2,0.993000,['Mamie Smith'],0.647,163827,0.186,0,11m7laMUgmOKqI3oYzuhne,0.000018,0,0.5190,-12.098,1,Golfing Papa,4,1920,0.1740,97.600,0.6890,1920
3,0.000173,['Oscar Velazquez'],0.730,422087,0.798,0,19Lc5SfJJ5O1oaxY0fpwfh,0.801000,2,0.1280,-7.311,1,True House Music - Xavier Santos & Carlos Gomi...,17,1920-01-01,0.0425,127.997,0.0422,1920
4,0.295000,['Mixe'],0.704,165224,0.707,1,2hJjbsLCytGsnAHfdsLejp,0.000246,10,0.4020,-6.036,0,Xuniverxe,2,1920-10-01,0.0768,122.076,0.2990,1920


In [5]:
# Data exploration
data.info()
data.isnull().sum()
df = data.drop(columns=['id', 'name', 'artists', 'release_date', 'year'])
df.corr()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174389 entries, 0 to 174388
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   acousticness      174389 non-null  float64
 1   artists           174389 non-null  object 
 2   danceability      174389 non-null  float64
 3   duration_ms       174389 non-null  int64  
 4   energy            174389 non-null  float64
 5   explicit          174389 non-null  int64  
 6   id                174389 non-null  object 
 7   instrumentalness  174389 non-null  float64
 8   key               174389 non-null  int64  
 9   liveness          174389 non-null  float64
 10  loudness          174389 non-null  float64
 11  mode              174389 non-null  int64  
 12  name              174389 non-null  object 
 13  popularity        174389 non-null  int64  
 14  release_date      174389 non-null  object 
 15  speechiness       174389 non-null  float64
 16  tempo             17

,acousticness,danceability,duration_ms,energy,explicit,instrumentalness,key,liveness,loudness,mode,popularity,speechiness,tempo,valence
acousticness,1.000000,-0.263217,-0.089169,-0.750852,-0.208176,0.221956,-0.028028,-0.029654,-0.546639,0.064633,-0.396744,-0.022437,-0.223840,-0.166968
danceability,-0.263217,1.000000,-0.100757,0.204838,0.200842,-0.215589,0.026266,-0.110033,0.249541,-0.048358,0.123746,0.239962,0.005479,0.536713
duration_ms,-0.089169,-0.100757,1.000000,0.060516,-0.033808,0.103621,0.002020,0.028942,0.019791,-0.046849,0.024717,-0.097838,-0.008182,-0.183199
energy,-0.750852,0.204838,0.060516,1.000000,0.102561,-0.177750,0.035780,0.134815,0.779267,-0.056160,0.328939,-0.112616,0.266448,0.326418
explicit,-0.208176,0.200842,-0.033808,0.102561,1.000000,-0.130609,0.005282,0.037288,0.106249,-0.062503,0.152545,0.353872,0.008075,-0.009275
instrumentalness,0.221956,-0.215589,0.103621,-0.177750,-0.130609,1.000000,-0.004619,-0.047941,-0.317562,-0.056731,-0.300625,-0.133966,-0.068656,-0.219188
key,-0.028028,0.026266,0.002020,0.035780,0.005282,-0.004619,1.000000,-0.003368,0.025227,-0.127397,0.001951,0.009648,0.005009,0.025592
liveness,-0.029654,-0.110033,0.028942,0.134815,0.037288,-0.047941,-0.003368,1.000000,0.062695,0.001677,-0.078959,0.122034,0.008586,-0.005781
loudness,-0.546639,0.249541,0.019791,0.779267,0.106249,-0.317562,0.025227,0.062695,1.000000,-0.019250,0.337194,-0.213504,0.217914,0.302520
mode,0.064633,-0.048358,-0.046849,-0.056160,-0.062503,-0.056731,-0.127397,0.001677,-0.019250,1.000000,0.007652,-0.040711,0.002438,0.021592


In [6]:
# Data transformation
from sklearn.preprocessing import MinMaxScaler
datatypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
normarization = data.select_dtypes(include=datatypes)
for col in normarization.columns:
    MinMaxScaler(col)

from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=10)
features = kmeans.fit_predict(normarization)
data['features'] = features
MinMaxScaler(data['features'])

MinMaxScaler(feature_range=0         4
1         4
2         4
3         3
4         4
         ..
174384    4
174385    4
174386    0
174387    0
174388    0
Name: features, Length: 174389, dtype: int32)

In [7]:
# Spotify recommender
class Spotify_Recommendation():
    def __init__(self, dataset):
        self.dataset = dataset
    def recommend(self, songs, amount=1):
        distance = []
        song = self.dataset[(self.dataset.name.str.lower() == songs.lower())].head(1).values[0]
        rec = self.dataset[self.dataset.name.str.lower() != songs.lower()]
        for songs in tqdm(rec.values):
            d = 0
            for col in np.arange(len(rec.columns)):
                if not col in [1, 6, 12, 14, 18]:
                    d = d + np.absolute(float(song[col]) - float(songs[col]))
            distance.append(d)
        rec['distance'] = distance
        rec = rec.sort_values('distance')
        columns = ['artists', 'name']
        return rec[columns][:amount]

recommendations = Spotify_Recommendation(data)
recommendations.recommend("Lovers Rock", 10)

100%|██████████| 174387/174387 [00:05<00:00, 30477.36it/s]
<ipython-input-7-7499836bba9e>:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rec['distance'] = distance


,artists,name
103171,['Barão Vermelho'],Bete Balanço
55318,['Shinedown'],Save Me
16385,['O-Zone'],Dragostea Din Tei
11168,['Bob Marley & The Wailers'],Positive Vibration
158441,"[""Olivia O'Brien""]",Love Myself
54226,"['Naughty By Nature', 'Zhané']",Jamboree (feat. Zhané)
85047,['The Outlaws'],Song For You
50644,['The Alan Parsons Project'],Mammagamma - Instrumental
107163,['Britney Spears'],My Prerogative
35098,['Los Askis'],¡Ay! El Amor
